# Historical data-construction source
Scientific status: historical. Static inspection only; not executed or validated anew.
All original outputs and execution metadata removed. Personal paths replaced. This is not an authorized runnable workflow.


# Data Cleaning for Main Data

## Overview

This notebook cleans the 2017 HMDA loan data and prepares final datasets for modeling. The original file is very large, so most steps read and write the data in chunks. The work includes column cleaning, row filtering, target creation, invalid value checks, feature checks, final column selection, and sampling for classification and regression.

## Step 1: Drop Unneeded Columns

In this step, the code removes columns that are not useful for modeling. Some columns have too many missing values, some are administrative fields, and some can create data leakage because they are known only after the loan decision. The file is read in chunks to save memory. After this step, the dataset has 51 columns and 14,285,496 rows.

In [ ]:
import pandas as pd
from pathlib import Path

# Input and output paths
DATA_PATH = Path("SOURCE_ROOT")
OUTPUT_PATH = Path("hmda_2017_cleaned_columns_only.csv")

CHUNK_SIZE = 200_000

# Columns to drop based on high missingness, leakage risk, or useless administrative meaning
columns_to_drop = [
    # Administrative / data quality columns
    "edit_status_name",
    "edit_status",
    "sequence_number",
    "application_date_indicator",

    # Applicant race detail columns with extremely high missingness
    "applicant_race_name_2",
    "applicant_race_2",
    "applicant_race_name_3",
    "applicant_race_3",
    "applicant_race_name_4",
    "applicant_race_4",
    "applicant_race_name_5",
    "applicant_race_5",

    # Co-applicant race detail columns with extremely high missingness
    "co_applicant_race_name_2",
    "co_applicant_race_2",
    "co_applicant_race_name_3",
    "co_applicant_race_3",
    "co_applicant_race_name_4",
    "co_applicant_race_4",
    "co_applicant_race_name_5",
    "co_applicant_race_5",

    # Denial reason columns are post-decision information and cause data leakage
    "denial_reason_name_1",
    "denial_reason_1",
    "denial_reason_name_2",
    "denial_reason_2",
    "denial_reason_name_3",
    "denial_reason_3",

    # Rate spread is mostly missing and related to loan pricing after the decision
    "rate_spread"
]

# --------------------------------------------------
# 1. Read only the header to check original columns
# --------------------------------------------------
header_df = pd.read_csv(
    DATA_PATH,
    nrows=0,
    dtype=str,
    keep_default_na=False
)

original_columns = header_df.columns.tolist()
original_column_count = len(original_columns)

existing_columns_to_drop = [
    col for col in columns_to_drop
    if col in original_columns
]

columns_not_found = [
    col for col in columns_to_drop
    if col not in original_columns
]

remaining_columns = [
    col for col in original_columns
    if col not in existing_columns_to_drop
]

print("Original column count:", original_column_count)
print("Columns selected for dropping:", len(existing_columns_to_drop))
print("Expected final column count:", len(remaining_columns))

print("\nColumns that will be dropped:")
for col in existing_columns_to_drop:
    print("-", col)

if columns_not_found:
    print("\nColumns requested for dropping but not found:")
    for col in columns_not_found:
        print("-", col)

# --------------------------------------------------
# 2. Process the full dataset in chunks
# --------------------------------------------------
first_chunk = True
processed_rows = 0

for chunk in pd.read_csv(
    DATA_PATH,
    dtype=str,
    keep_default_na=False,
    na_filter=False,
    chunksize=CHUNK_SIZE
):
    # Drop selected columns
    chunk = chunk.drop(columns=existing_columns_to_drop)

    # Write the first chunk with header, then append the rest without header
    chunk.to_csv(
        OUTPUT_PATH,
        index=False,
        mode="w" if first_chunk else "a",
        header=first_chunk
    )

    first_chunk = False
    processed_rows += len(chunk)

    print(f"Processed rows: {processed_rows:,} | Current column count: {chunk.shape[1]}")

print("\nDone.")
print("Output saved to:", OUTPUT_PATH.resolve())

# --------------------------------------------------
# 3. Check output file column count
# --------------------------------------------------
output_header_df = pd.read_csv(
    OUTPUT_PATH,
    nrows=0,
    dtype=str,
    keep_default_na=False
)

output_columns = output_header_df.columns.tolist()

print("\nFinal check:")
print("Original column count:", original_column_count)
print("Dropped column count:", len(existing_columns_to_drop))
print("Output column count:", len(output_columns))

if len(output_columns) == len(remaining_columns):
    print("Column count check: PASSED")
else:
    print("Column count check: FAILED")

# --------------------------------------------------
# 4. Optional: check row count of output file
# --------------------------------------------------
output_row_count = 0

for chunk in pd.read_csv(
    OUTPUT_PATH,
    dtype=str,
    keep_default_na=False,
    na_filter=False,
    chunksize=CHUNK_SIZE
):
    output_row_count += len(chunk)

print("\nOutput row count:", f"{output_row_count:,}")

## Step 2: Remove Rows with Missing Important Values

This step removes rows that have missing or invalid values in important columns. These columns include income, loan amount, location, tract information, and population information. The code also treats values like empty strings, `NA`, `null`, `.`, and `-` as missing. After this step, 10,799,874 rows remain.

In [ ]:
import pandas as pd
from pathlib import Path

# Input and output paths
DATA_PATH = Path("SOURCE_ROOT")
OUTPUT_PATH = Path("hmda_2017_cleaned_columns_and_rows.csv")

CHUNK_SIZE = 200_000

# Rows with missing values in these columns will be dropped
required_non_missing_columns = [
    "applicant_income_000s",
    "msamd_name",
    "msamd",
    "tract_to_msamd_income",
    "number_of_owner_occupied_units",
    "number_of_1_to_4_family_units",
    "census_tract_number",
    "population",
    "minority_population",
    "hud_median_family_income",
    "county_name",
    "county_code",
    "state_name",
    "state_abbr",
    "state_code",
    "loan_amount_000s"
]

# Values that should be treated as missing
missing_like_values = {
    "",
    " ",
    "nan", "NaN", "NAN",
    "none", "None", "NONE",
    "null", "Null", "NULL",
    "na", "NA", "N/A", "n/a",
    ".", "-", "--"
}

# --------------------------------------------------
# 1. Check available columns
# --------------------------------------------------
header_df = pd.read_csv(
    DATA_PATH,
    nrows=0,
    dtype=str,
    keep_default_na=False,
    na_filter=False
)

available_columns = header_df.columns.tolist()

existing_required_columns = [
    col for col in required_non_missing_columns
    if col in available_columns
]

missing_required_columns = [
    col for col in required_non_missing_columns
    if col not in available_columns
]

print("Required columns found:", len(existing_required_columns))
print(existing_required_columns)

if missing_required_columns:
    print("\nRequired columns not found in file:")
    for col in missing_required_columns:
        print("-", col)

# --------------------------------------------------
# 2. Process full dataset chunk by chunk
# --------------------------------------------------
first_chunk = True
total_rows_before = 0
total_rows_after = 0
removed_rows_total = 0

for chunk_id, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    rows_before = len(chunk)
    total_rows_before += rows_before

    # Strip whitespace in required columns only
    for col in existing_required_columns:
        chunk[col] = chunk[col].astype(str).str.strip()

    # Build mask for rows that have missing-like values in any required column
    missing_mask = pd.Series(False, index=chunk.index)

    for col in existing_required_columns:
        missing_mask = missing_mask | chunk[col].isin(missing_like_values)

    # Keep only rows without missing values in required columns
    chunk_clean = chunk.loc[~missing_mask].copy()

    rows_after = len(chunk_clean)
    removed_rows = rows_before - rows_after

    total_rows_after += rows_after
    removed_rows_total += removed_rows

    # Write clean chunk to output file
    chunk_clean.to_csv(
        OUTPUT_PATH,
        index=False,
        mode="w" if first_chunk else "a",
        header=first_chunk
    )

    first_chunk = False

    print(
        f"Chunk {chunk_id} | "
        f"Before: {rows_before:,} | "
        f"After: {rows_after:,} | "
        f"Removed: {removed_rows:,} | "
        f"Total kept: {total_rows_after:,}"
    )

print("\nDone.")
print("Output saved to:", OUTPUT_PATH.resolve())
print("Total rows before:", f"{total_rows_before:,}")
print("Total rows after:", f"{total_rows_after:,}")
print("Total rows removed:", f"{removed_rows_total:,}")
print("Removed percent:", round(removed_rows_total / total_rows_before * 100, 2), "%")

## Step 3: Create the Classification Target

This step keeps only loan actions that can be used for binary classification. Actions `1`, `2`, and `8` are treated as approved loans, and actions `3` and `7` are treated as rejected loans. A new column named `loan_approved` is created, where `1` means approved and `0` means rejected. After this step, 8,074,536 rows remain.

In [ ]:
import pandas as pd
from pathlib import Path

# Input and output paths
DATA_PATH = Path("SOURCE_ROOT")
OUTPUT_PATH = Path("hmda_2017_classification_ready.csv")

CHUNK_SIZE = 200_000

# Action values to keep
approved_actions = {"1", "2", "8"}
rejected_actions = {"3", "7"}
valid_actions = approved_actions | rejected_actions

# --------------------------------------------------
# Process full dataset chunk by chunk
# --------------------------------------------------
first_chunk = True
total_rows_before = 0
total_rows_after = 0
removed_rows_total = 0

for chunk_id, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    rows_before = len(chunk)
    total_rows_before += rows_before

    # Clean action_taken values
    chunk["action_taken"] = chunk["action_taken"].astype(str).str.strip()

    # Keep only action_taken values 1, 2, 3, 7, 8
    chunk = chunk[chunk["action_taken"].isin(valid_actions)].copy()

    # Create binary target column
    chunk["loan_approved"] = chunk["action_taken"].map(
        {
            "1": 1,
            "2": 1,
            "8": 1,
            "3": 0,
            "7": 0
        }
    )

    rows_after = len(chunk)
    removed_rows = rows_before - rows_after

    total_rows_after += rows_after
    removed_rows_total += removed_rows

    # Write cleaned chunk to output file
    chunk.to_csv(
        OUTPUT_PATH,
        index=False,
        mode="w" if first_chunk else "a",
        header=first_chunk
    )

    first_chunk = False

    print(
        f"Chunk {chunk_id} | "
        f"Before: {rows_before:,} | "
        f"After: {rows_after:,} | "
        f"Removed: {removed_rows:,} | "
        f"Total kept: {total_rows_after:,}"
    )

print("\nDone.")
print("Output saved to:", OUTPUT_PATH.resolve())
print("Total rows before:", f"{total_rows_before:,}")
print("Total rows after:", f"{total_rows_after:,}")
print("Total rows removed:", f"{removed_rows_total:,}")
print("Removed percent:", round(removed_rows_total / total_rows_before * 100, 2), "%")

## Step 4: Remove Duplicate Rows

This step removes duplicate rows from the classification-ready file. Before checking duplicates, the code strips extra spaces from text values. Then it creates a hash for each row and keeps only rows that appear for the first time. After this step, 8,063,000 unique rows remain.

In [ ]:
import pandas as pd
from pathlib import Path

# Input and output paths
DATA_PATH = Path("SOURCE_ROOT")
OUTPUT_PATH = Path("hmda_2017_no_duplicates.csv")

CHUNK_SIZE = 200_000

# This set stores hashes of rows that have already been seen
seen_hashes = set()

first_chunk = True
total_rows_before = 0
total_rows_after = 0
total_duplicates_removed = 0

for chunk_id, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    rows_before = len(chunk)
    total_rows_before += rows_before

    # Normalize string values before hashing
    # This helps avoid treating " value " and "value" as different rows
    chunk = chunk.astype(str)
    chunk = chunk.apply(lambda col: col.str.strip())

    # Create a stable row hash based on all remaining columns
    row_hashes = pd.util.hash_pandas_object(
        chunk,
        index=False
    ).astype("uint64")

    # Keep rows whose hash has not been seen before
    keep_mask = []

    for h in row_hashes:
        h_int = int(h)

        if h_int in seen_hashes:
            keep_mask.append(False)
        else:
            keep_mask.append(True)
            seen_hashes.add(h_int)

    chunk_unique = chunk.loc[keep_mask].copy()

    rows_after = len(chunk_unique)
    duplicates_removed = rows_before - rows_after

    total_rows_after += rows_after
    total_duplicates_removed += duplicates_removed

    # Write unique rows to output file
    chunk_unique.to_csv(
        OUTPUT_PATH,
        index=False,
        mode="w" if first_chunk else "a",
        header=first_chunk
    )

    first_chunk = False

    print(
        f"Chunk {chunk_id} | "
        f"Before: {rows_before:,} | "
        f"After: {rows_after:,} | "
        f"Removed duplicates: {duplicates_removed:,} | "
        f"Total kept: {total_rows_after:,}"
    )

print("\nDone.")
print("Output saved to:", OUTPUT_PATH.resolve())
print("Total rows before:", f"{total_rows_before:,}")
print("Total rows after:", f"{total_rows_after:,}")
print("Total duplicates removed:", f"{total_duplicates_removed:,}")
print("Removed percent:", round(total_duplicates_removed / total_rows_before * 100, 4), "%")
print("Unique hashes stored:", f"{len(seen_hashes):,}")

## Step 5: Check Numeric Columns

This step studies the main numeric columns, such as loan amount, applicant income, population, and income-related tract values. The code converts these columns to numbers and creates a summary table with missing counts, minimum, maximum, mean, median, quartiles, and standard deviation. This helps us understand the scale and possible outliers before modeling.

In [ ]:
import pandas as pd
from pathlib import Path

# Input file
DATA_PATH = Path("hmda_2017_classification_ready.csv")

CHUNK_SIZE = 200_000

# Continuous / semi-continuous numeric columns
continuous_numeric_cols = [
    "loan_amount_000s",
    "applicant_income_000s",
    "population",
    "minority_population",
    "hud_median_family_income",
    "tract_to_msamd_income",
    "number_of_owner_occupied_units",
    "number_of_1_to_4_family_units"
]

# --------------------------------------------------
# 1. Check which columns exist in the file
# --------------------------------------------------
header_df = pd.read_csv(
    DATA_PATH,
    nrows=0,
    dtype=str,
    keep_default_na=False,
    na_filter=False
)

existing_numeric_cols = [
    col for col in continuous_numeric_cols
    if col in header_df.columns
]

missing_numeric_cols = [
    col for col in continuous_numeric_cols
    if col not in header_df.columns
]

print("Existing numeric columns:")
print(existing_numeric_cols)

if missing_numeric_cols:
    print("\nColumns not found:")
    print(missing_numeric_cols)

# --------------------------------------------------
# 2. Read the file chunk by chunk
#    Store only numeric columns, then make one final summary
# --------------------------------------------------
numeric_chunks = []

total_rows = 0

for chunk_id, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    total_rows += len(chunk)

    # Convert selected columns to numeric
    numeric_chunk = chunk[existing_numeric_cols].apply(
        pd.to_numeric,
        errors="coerce"
    )

    numeric_chunks.append(numeric_chunk)

    print(f"Processed chunk {chunk_id} | Total rows processed: {total_rows:,}")

# --------------------------------------------------
# 3. Combine numeric chunks and create one final summary
# --------------------------------------------------
df_numeric_all = pd.concat(numeric_chunks, ignore_index=True)

numeric_summary = df_numeric_all.describe(
    percentiles=[0.25, 0.5, 0.75]
).T

numeric_summary = numeric_summary[
    ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
].rename(columns={
    "count": "non_missing_count",
    "mean": "mean",
    "std": "std",
    "min": "min",
    "25%": "q1_25%",
    "50%": "median_50%",
    "75%": "q3_75%",
    "max": "max"
})

# Add missing counts
numeric_summary["missing_count"] = df_numeric_all.isna().sum()
numeric_summary["missing_percent"] = (
    df_numeric_all.isna().mean() * 100
).round(2)

# Reorder columns
numeric_summary = numeric_summary[
    [
        "non_missing_count",
        "missing_count",
        "missing_percent",
        "min",
        "q1_25%",
        "median_50%",
        "mean",
        "q3_75%",
        "max",
        "std"
    ]
]

# Optional: round values for readability
numeric_summary = numeric_summary.round(3)

print("\nTotal rows processed:", f"{total_rows:,}")
display(numeric_summary)

## Step 6: Remove Invalid Values

This step makes a stricter clean version of the classification dataset. It removes rows with real missing values, invalid zero values, non-numeric values in numeric columns, invalid numeric ranges, and wrong categorical codes. It also keeps special text values when they have a real meaning, such as `Not applicable`. After this step, 8,064,813 rows remain, and a problem summary file is saved.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# 0. File paths
# =========================================================

DATA_PATH = Path("SOURCE_ROOT")
OUTPUT_PATH = Path("hmda_2017_classification_ready_cleaned_invalid_removed.csv")

CHUNK_SIZE = 200_000

# =========================================================
# 1. Missing-like values
# =========================================================

real_missing_values = {
    "",
    "nan", "NaN", "NAN",
    "none", "None", "NONE",
    "null", "Null", "NULL",
    "na", "NA", "N/A", "n/a",
    ".", "-", "--"
}

real_missing_values_lower = {v.lower() for v in real_missing_values}

# These values are meaningful categories, not real missing values.
special_meaning_values = {
    "Information not provided by applicant in mail, Internet, or telephone application",
    "Information not provided",
    "Not applicable",
    "No co-applicant"
}

# =========================================================
# 2. Numeric columns
# =========================================================

numeric_value_cols = [
    "loan_amount_000s",
    "applicant_income_000s",
    "population",
    "minority_population",
    "hud_median_family_income",
    "tract_to_msamd_income",
    "number_of_owner_occupied_units",
    "number_of_1_to_4_family_units"
]

# Zero is invalid / missing-like in these columns.
zero_should_not_exist_cols = [
    "loan_amount_000s",
    "applicant_income_000s",
    "population",
    "hud_median_family_income",
    "tract_to_msamd_income",
    "number_of_owner_occupied_units",
    "number_of_1_to_4_family_units"
]

# Zero is valid in these columns.
zero_is_valid_cols = [
    "minority_population",
    "loan_approved"
]

# =========================================================
# 3. Valid categorical codes
# =========================================================

valid_codes = {
    "agency_code": {"1", "2", "3", "5", "7", "9"},
    "loan_type": {"1", "2", "3", "4"},
    "property_type": {"1", "2", "3"},
    "loan_purpose": {"1", "2", "3"},
    "owner_occupancy": {"1", "2", "3"},
    "preapproval": {"1", "2", "3"},

    # If action_taken still exists, only keep the actions used in your classification task.
    "action_taken": {"1", "2", "3", "7", "8"},

    "applicant_ethnicity": {"1", "2", "3", "4"},
    "co_applicant_ethnicity": {"1", "2", "3", "4", "5"},

    "applicant_race_1": {"1", "2", "3", "4", "5", "6", "7"},
    "co_applicant_race_1": {"1", "2", "3", "4", "5", "6", "7", "8"},

    "applicant_sex": {"1", "2", "3", "4"},
    "co_applicant_sex": {"1", "2", "3", "4", "5"},

    "hoepa_status": {"1", "2"},
    "lien_status": {"1", "2", "3", "4"},

    # Final binary target
    "loan_approved": {"0", "1"}
}

# =========================================================
# 4. Leftover columns that should not be used if they still exist
# =========================================================

columns_to_drop_if_still_exist = [
    "edit_status_name",
    "edit_status",
    "sequence_number",
    "application_date_indicator",

    "applicant_race_name_2",
    "applicant_race_2",
    "applicant_race_name_3",
    "applicant_race_3",
    "applicant_race_name_4",
    "applicant_race_4",
    "applicant_race_name_5",
    "applicant_race_5",

    "co_applicant_race_name_2",
    "co_applicant_race_2",
    "co_applicant_race_name_3",
    "co_applicant_race_3",
    "co_applicant_race_name_4",
    "co_applicant_race_4",
    "co_applicant_race_name_5",
    "co_applicant_race_5",

    "denial_reason_name_1",
    "denial_reason_1",
    "denial_reason_name_2",
    "denial_reason_2",
    "denial_reason_name_3",
    "denial_reason_3",

    "rate_spread",
    "purchaser_type_name",
    "purchaser_type",

    # If you already created loan_approved, these should not remain as model features.
    "action_taken_name"
]

# =========================================================
# 5. Helper functions
# =========================================================

def is_real_missing(s):
    """
    Detect real missing-like values after converting to string.
    """
    temp = s.astype(str).str.strip().str.lower()
    return temp.isin(real_missing_values_lower)


def is_zero_like(s):
    """
    Detect values like 0, 0.0, 0.00, 000, 000.0.
    """
    temp = s.astype(str).str.strip()

    zero_text_mask = temp.str.fullmatch(r"0+(\.0+)?", na=False)

    numeric_temp = pd.to_numeric(temp, errors="coerce")
    zero_numeric_mask = numeric_temp.eq(0)

    return zero_text_mask | zero_numeric_mask


def init_col_stats():
    """
    Initialize per-column problem counters.
    """
    return {
        "real_missing_count": 0,
        "zero_as_missing_count": 0,
        "non_numeric_count": 0,
        "invalid_numeric_range_count": 0,
        "invalid_code_count": 0
    }


# =========================================================
# 6. Read header and prepare dynamic column lists
# =========================================================

header_df = pd.read_csv(
    DATA_PATH,
    nrows=0,
    dtype=str,
    keep_default_na=False,
    na_filter=False
)

original_columns = header_df.columns.tolist()

existing_drop_cols = [
    col for col in columns_to_drop_if_still_exist
    if col in original_columns
]

remaining_columns_after_drop = [
    col for col in original_columns
    if col not in existing_drop_cols
]

existing_numeric_cols = [
    col for col in numeric_value_cols
    if col in remaining_columns_after_drop
]

existing_zero_invalid_cols = [
    col for col in zero_should_not_exist_cols
    if col in remaining_columns_after_drop
]

existing_valid_code_cols = {
    col: codes
    for col, codes in valid_codes.items()
    if col in remaining_columns_after_drop
}

print("Original column count:", len(original_columns))
print("Columns to drop if still present:", len(existing_drop_cols))
print("Expected output column count:", len(remaining_columns_after_drop))

print("\nColumns dropped before row cleaning:")
for col in existing_drop_cols:
    print("-", col)

print("\nNumeric columns checked:")
print(existing_numeric_cols)

print("\nZero-invalid columns checked:")
print(existing_zero_invalid_cols)

print("\nCategorical code columns checked:")
print(list(existing_valid_code_cols.keys()))

# =========================================================
# 7. Process file chunk by chunk
# =========================================================

first_chunk = True

total_rows_before = 0
total_rows_after = 0
total_rows_removed = 0

reason_counts = {
    "real_missing_rows": 0,
    "zero_as_missing_rows": 0,
    "non_numeric_rows": 0,
    "invalid_numeric_range_rows": 0,
    "invalid_code_rows": 0,
    "total_removed_rows_unique": 0
}

per_column_stats = {}

def add_col_stat(col, key, count):
    if col not in per_column_stats:
        per_column_stats[col] = init_col_stats()
    per_column_stats[col][key] += int(count)


for chunk_id, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    rows_before = len(chunk)
    total_rows_before += rows_before

    # Drop leftover columns if they still exist.
    drop_cols_in_chunk = [
        col for col in existing_drop_cols
        if col in chunk.columns
    ]

    if drop_cols_in_chunk:
        chunk = chunk.drop(columns=drop_cols_in_chunk)

    # Normalize all remaining columns as stripped strings.
    for col in chunk.columns:
        chunk[col] = chunk[col].astype(str).str.strip()

    # -----------------------------------------------------
    # A. Real missing values in any remaining column
    # -----------------------------------------------------
    real_missing_row_mask = pd.Series(False, index=chunk.index)

    for col in chunk.columns:
        col_missing_mask = is_real_missing(chunk[col])
        add_col_stat(col, "real_missing_count", col_missing_mask.sum())
        real_missing_row_mask = real_missing_row_mask | col_missing_mask

    # -----------------------------------------------------
    # B. Zero-like values in columns where zero is invalid
    # -----------------------------------------------------
    zero_as_missing_row_mask = pd.Series(False, index=chunk.index)

    for col in existing_zero_invalid_cols:
        if col in chunk.columns:
            col_zero_mask = is_zero_like(chunk[col]) & (~is_real_missing(chunk[col]))
            add_col_stat(col, "zero_as_missing_count", col_zero_mask.sum())
            zero_as_missing_row_mask = zero_as_missing_row_mask | col_zero_mask

    # -----------------------------------------------------
    # C. Non-numeric values and invalid numeric ranges
    # -----------------------------------------------------
    non_numeric_row_mask = pd.Series(False, index=chunk.index)
    invalid_numeric_range_row_mask = pd.Series(False, index=chunk.index)

    for col in existing_numeric_cols:
        if col not in chunk.columns:
            continue

        col_missing_mask = is_real_missing(chunk[col])
        numeric_s = pd.to_numeric(chunk[col], errors="coerce")

        # Non-numeric but not already missing-like.
        col_non_numeric_mask = numeric_s.isna() & (~col_missing_mask)

        add_col_stat(col, "non_numeric_count", col_non_numeric_mask.sum())
        non_numeric_row_mask = non_numeric_row_mask | col_non_numeric_mask

        # Range rules for numeric columns.
        if col == "minority_population":
            # Minority percentage must be between 0 and 100.
            col_invalid_range_mask = (
                numeric_s.notna()
                & ((numeric_s < 0) | (numeric_s > 100))
            )
        elif col in zero_should_not_exist_cols:
            # These columns must be strictly positive.
            col_invalid_range_mask = (
                numeric_s.notna()
                & (numeric_s <= 0)
            )
        else:
            # General numeric columns should not be negative.
            col_invalid_range_mask = (
                numeric_s.notna()
                & (numeric_s < 0)
            )

        add_col_stat(col, "invalid_numeric_range_count", col_invalid_range_mask.sum())
        invalid_numeric_range_row_mask = invalid_numeric_range_row_mask | col_invalid_range_mask

    # -----------------------------------------------------
    # D. Invalid categorical codes
    # -----------------------------------------------------
    invalid_code_row_mask = pd.Series(False, index=chunk.index)

    for col, allowed_codes in existing_valid_code_cols.items():
        if col not in chunk.columns:
            continue

        col_missing_mask = is_real_missing(chunk[col])

        col_invalid_code_mask = (
            (~col_missing_mask)
            & (~chunk[col].isin(allowed_codes))
        )

        add_col_stat(col, "invalid_code_count", col_invalid_code_mask.sum())
        invalid_code_row_mask = invalid_code_row_mask | col_invalid_code_mask

    # -----------------------------------------------------
    # E. Combine all row-removal rules
    # -----------------------------------------------------
    rows_to_remove_mask = (
        real_missing_row_mask
        | zero_as_missing_row_mask
        | non_numeric_row_mask
        | invalid_numeric_range_row_mask
        | invalid_code_row_mask
    )

    chunk_clean = chunk.loc[~rows_to_remove_mask].copy()

    rows_after = len(chunk_clean)
    rows_removed = rows_before - rows_after

    total_rows_after += rows_after
    total_rows_removed += rows_removed

    reason_counts["real_missing_rows"] += int(real_missing_row_mask.sum())
    reason_counts["zero_as_missing_rows"] += int(zero_as_missing_row_mask.sum())
    reason_counts["non_numeric_rows"] += int(non_numeric_row_mask.sum())
    reason_counts["invalid_numeric_range_rows"] += int(invalid_numeric_range_row_mask.sum())
    reason_counts["invalid_code_rows"] += int(invalid_code_row_mask.sum())
    reason_counts["total_removed_rows_unique"] += int(rows_to_remove_mask.sum())

    # Write cleaned chunk.
    chunk_clean.to_csv(
        OUTPUT_PATH,
        index=False,
        mode="w" if first_chunk else "a",
        header=first_chunk
    )

    first_chunk = False

    print(
        f"Chunk {chunk_id} | "
        f"Before: {rows_before:,} | "
        f"After: {rows_after:,} | "
        f"Removed: {rows_removed:,} | "
        f"Total kept: {total_rows_after:,}"
    )

# =========================================================
# 8. Final summary
# =========================================================

print("\nDone.")
print("Output saved to:", OUTPUT_PATH.resolve())

print("\nFinal row summary:")
print("Total rows before:", f"{total_rows_before:,}")
print("Total rows after:", f"{total_rows_after:,}")
print("Total rows removed:", f"{total_rows_removed:,}")

if total_rows_before > 0:
    print("Removed percent:", round(total_rows_removed / total_rows_before * 100, 4), "%")

print("\nRemoval reason counts:")
for key, value in reason_counts.items():
    print(f"{key}: {value:,}")

# Per-column problem summary
problem_summary = pd.DataFrame.from_dict(
    per_column_stats,
    orient="index"
).reset_index().rename(columns={"index": "column"})

problem_summary["total_problem_count"] = (
    problem_summary["real_missing_count"]
    + problem_summary["zero_as_missing_count"]
    + problem_summary["non_numeric_count"]
    + problem_summary["invalid_numeric_range_count"]
    + problem_summary["invalid_code_count"]
)

problem_summary = problem_summary.sort_values(
    "total_problem_count",
    ascending=False
).reset_index(drop=True)

print("\nTop columns with detected problems:")
display(problem_summary.head(30))

# Save the cleaning report.
REPORT_PATH = Path("hmda_2017_cleaning_problem_summary.csv")
problem_summary.to_csv(REPORT_PATH, index=False)

print("\nProblem summary saved to:", REPORT_PATH.resolve())

## Step 7: Check Numeric Columns After Invalid Value Cleaning

This step repeats the numeric summary after invalid rows are removed. It checks that the main numeric columns have no missing values and shows their minimum, maximum, mean, median, quartiles, and standard deviation. This confirms that the stricter cleaning step worked correctly.

In [ ]:
import pandas as pd
from pathlib import Path

# Input file
DATA_PATH = Path("hmda_2017_classification_ready_cleaned_invalid_removed.csv")

CHUNK_SIZE = 200_000

# Continuous / semi-continuous numeric columns
continuous_numeric_cols = [
    "loan_amount_000s",
    "applicant_income_000s",
    "population",
    "minority_population",
    "hud_median_family_income",
    "tract_to_msamd_income",
    "number_of_owner_occupied_units",
    "number_of_1_to_4_family_units"
]

# --------------------------------------------------
# 1. Check which columns exist in the file
# --------------------------------------------------
header_df = pd.read_csv(
    DATA_PATH,
    nrows=0,
    dtype=str,
    keep_default_na=False,
    na_filter=False
)

existing_numeric_cols = [
    col for col in continuous_numeric_cols
    if col in header_df.columns
]

missing_numeric_cols = [
    col for col in continuous_numeric_cols
    if col not in header_df.columns
]

print("Existing numeric columns:")
print(existing_numeric_cols)

if missing_numeric_cols:
    print("\nColumns not found:")
    print(missing_numeric_cols)

# --------------------------------------------------
# 2. Read the file chunk by chunk
#    Store only numeric columns, then make one final summary
# --------------------------------------------------
numeric_chunks = []

total_rows = 0

for chunk_id, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    total_rows += len(chunk)

    # Convert selected columns to numeric
    numeric_chunk = chunk[existing_numeric_cols].apply(
        pd.to_numeric,
        errors="coerce"
    )

    numeric_chunks.append(numeric_chunk)

    print(f"Processed chunk {chunk_id} | Total rows processed: {total_rows:,}")

# --------------------------------------------------
# 3. Combine numeric chunks and create one final summary
# --------------------------------------------------
df_numeric_all = pd.concat(numeric_chunks, ignore_index=True)

numeric_summary = df_numeric_all.describe(
    percentiles=[0.25, 0.5, 0.75]
).T

numeric_summary = numeric_summary[
    ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
].rename(columns={
    "count": "non_missing_count",
    "mean": "mean",
    "std": "std",
    "min": "min",
    "25%": "q1_25%",
    "50%": "median_50%",
    "75%": "q3_75%",
    "max": "max"
})

# Add missing counts
numeric_summary["missing_count"] = df_numeric_all.isna().sum()
numeric_summary["missing_percent"] = (
    df_numeric_all.isna().mean() * 100
).round(2)

# Reorder columns
numeric_summary = numeric_summary[
    [
        "non_missing_count",
        "missing_count",
        "missing_percent",
        "min",
        "q1_25%",
        "median_50%",
        "mean",
        "q3_75%",
        "max",
        "std"
    ]
]

# Optional: round values for readability
numeric_summary = numeric_summary.round(3)

print("\nTotal rows processed:", f"{total_rows:,}")
display(numeric_summary)

## Step 8: Find Extreme Income and Loan Values

This step looks for the 100 smallest and 100 largest values in `loan_amount_000s` and `applicant_income_000s`. These two columns are important because very small or very large values can affect the model. The data is processed in chunks, so only the running top and bottom values are stored.

In [ ]:
import pandas as pd
from pathlib import Path

# Input file
DATA_PATH = Path("hmda_2017_classification_ready_cleaned_invalid_removed.csv")

CHUNK_SIZE = 200_000

target_cols = [
    "loan_amount_000s",
    "applicant_income_000s"
]

TOP_N = 100

# Store running smallest and largest values
smallest_values = {col: pd.Series(dtype="float64") for col in target_cols}
largest_values = {col: pd.Series(dtype="float64") for col in target_cols}

total_rows_processed = 0

for chunk_id, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    total_rows_processed += len(chunk)

    for col in target_cols:
        if col not in chunk.columns:
            print(f"Column not found: {col}")
            continue

        # Convert to numeric
        numeric_s = pd.to_numeric(
            chunk[col].astype(str).str.strip(),
            errors="coerce"
        ).dropna()

        # Update smallest values
        smallest_values[col] = (
            pd.concat([smallest_values[col], numeric_s])
            .nsmallest(TOP_N)
            .reset_index(drop=True)
        )

        # Update largest values
        largest_values[col] = (
            pd.concat([largest_values[col], numeric_s])
            .nlargest(TOP_N)
            .reset_index(drop=True)
        )

    print(f"Processed chunk {chunk_id} | Total rows processed: {total_rows_processed:,}")

print("\nDone.")
print("Total rows processed:", f"{total_rows_processed:,}")

## Step 9: Display Extreme Values

This step displays the smallest and largest values found in the previous step. The result helps us see possible outliers, such as very small values equal to `1` and very large income or loan amounts. These values need attention before final modeling.

In [ ]:
for col in target_cols:
    print("=" * 80)
    print(f"Column: {col}")
    
    print("\n100 smallest values:")
    display(
        pd.DataFrame({
            col: smallest_values[col].sort_values(ascending=True).values
        })
    )

    print("\n100 largest values:")
    display(
        pd.DataFrame({
            col: largest_values[col].sort_values(ascending=False).values
        })
    )

## Step 10: Analyze Suspicious Extreme Values

This step studies important suspicious groups in more detail. It checks rows where income or loan amount equals `1`, which means 1,000 dollars, and also checks the top 100 highest income and loan rows. For each group, it compares the approval rate with the full dataset and summarizes loan amounts for approved loans.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# 0. File path and settings
# =========================================================

DATA_PATH = Path("hmda_2017_classification_ready_cleaned_invalid_removed.csv")

CHUNK_SIZE = 200_000
TOP_N = 100

income_col = "applicant_income_000s"
loan_col = "loan_amount_000s"
class_target_col = "loan_approved"

# Because these columns are in thousands of dollars:
# value == 1 means 1,000 dollars
ONE_THOUSAND_DOLLARS_VALUE = 1


# =========================================================
# 1. Helper functions
# =========================================================

def safe_to_numeric(s):
    """Convert a pandas Series to numeric after stripping strings."""
    return pd.to_numeric(
        s.astype(str).str.strip(),
        errors="coerce"
    )


def classification_distribution(df, target_col=class_target_col):
    """Return count and percent distribution of classification target."""
    if target_col not in df.columns or len(df) == 0:
        return pd.DataFrame(columns=[target_col, "count", "percent"])

    counts = (
        df[target_col]
        .astype(str)
        .str.strip()
        .value_counts(dropna=False)
        .sort_index()
    )

    result = counts.reset_index()
    result.columns = [target_col, "count"]
    result["percent"] = (result["count"] / result["count"].sum() * 100).round(2)

    return result


def regression_distribution_for_approved(df):
    """
    Return regression target summary for approved rows only.
    Regression target is loan_amount_000s.
    """
    if len(df) == 0 or class_target_col not in df.columns or loan_col not in df.columns:
        return pd.DataFrame([{
            "approved_rows_for_regression": 0,
            "min": np.nan,
            "q1_25%": np.nan,
            "median_50%": np.nan,
            "mean": np.nan,
            "q3_75%": np.nan,
            "max": np.nan,
            "std": np.nan
        }])

    approved_df = df[
        df[class_target_col].astype(str).str.strip() == "1"
    ].copy()

    y = safe_to_numeric(approved_df[loan_col]).dropna()

    if len(y) == 0:
        return pd.DataFrame([{
            "approved_rows_for_regression": 0,
            "min": np.nan,
            "q1_25%": np.nan,
            "median_50%": np.nan,
            "mean": np.nan,
            "q3_75%": np.nan,
            "max": np.nan,
            "std": np.nan
        }])

    return pd.DataFrame([{
        "approved_rows_for_regression": len(y),
        "min": y.min(),
        "q1_25%": y.quantile(0.25),
        "median_50%": y.quantile(0.50),
        "mean": y.mean(),
        "q3_75%": y.quantile(0.75),
        "max": y.max(),
        "std": y.std()
    }]).round(3)


def compare_group_to_overall_distribution(group_df, overall_distribution):
    """
    Create one table comparing the group's loan_approved distribution
    with the overall dataset distribution.
    """
    group_distribution = classification_distribution(group_df)

    comparison = pd.DataFrame({
        class_target_col: ["0", "1"]
    })

    comparison = comparison.merge(
        overall_distribution.rename(columns={
            "count": "overall_count",
            "percent": "overall_percent"
        }),
        on=class_target_col,
        how="left"
    )

    comparison = comparison.merge(
        group_distribution.rename(columns={
            "count": "group_count",
            "percent": "group_percent"
        }),
        on=class_target_col,
        how="left"
    )

    comparison = comparison.fillna(0)

    comparison["group_minus_overall_percent"] = (
        comparison["group_percent"] - comparison["overall_percent"]
    ).round(2)

    return comparison


def make_full_report(group_name, df_group, overall_distribution):
    """Create classification comparison and regression report for one group."""
    print("=" * 100)
    print(group_name)
    print("Rows in group:", f"{len(df_group):,}")

    print("\nClassification target distribution: group vs overall dataset")
    display(compare_group_to_overall_distribution(df_group, overall_distribution))

    print("\nRegression target distribution among approved rows only:")
    display(regression_distribution_for_approved(df_group))


# =========================================================
# 2. Storage for filtered groups and top-100 groups
# =========================================================

income_1000_parts = []
loan_1000_parts = []

top_income_df = pd.DataFrame()
top_loan_df = pd.DataFrame()

overall_target_counts = {}

total_rows_processed = 0


# =========================================================
# 3. Read data chunk by chunk
# =========================================================

for chunk_id, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    total_rows_processed += len(chunk)

    required_cols = [income_col, loan_col, class_target_col]
    missing_cols = [col for col in required_cols if col not in chunk.columns]

    if missing_cols:
        raise ValueError(f"Missing columns in dataset: {missing_cols}")

    # -----------------------------------------------------
    # Overall loan_approved distribution over the full dataset
    # -----------------------------------------------------
    target_counts = (
        chunk[class_target_col]
        .astype(str)
        .str.strip()
        .value_counts(dropna=False)
    )

    for key, value in target_counts.items():
        overall_target_counts[key] = overall_target_counts.get(key, 0) + int(value)

    # -----------------------------------------------------
    # Numeric conversion for income and loan amount
    # -----------------------------------------------------
    chunk["_income_numeric"] = safe_to_numeric(chunk[income_col])
    chunk["_loan_numeric"] = safe_to_numeric(chunk[loan_col])

    # -----------------------------------------------------
    # A. Rows with applicant_income_000s == 1
    # -----------------------------------------------------
    income_1000_chunk = chunk[
        chunk["_income_numeric"] == ONE_THOUSAND_DOLLARS_VALUE
    ].copy()

    if len(income_1000_chunk) > 0:
        income_1000_parts.append(
            income_1000_chunk.drop(columns=["_income_numeric", "_loan_numeric"])
        )

    # -----------------------------------------------------
    # B. Rows with loan_amount_000s == 1
    # -----------------------------------------------------
    loan_1000_chunk = chunk[
        chunk["_loan_numeric"] == ONE_THOUSAND_DOLLARS_VALUE
    ].copy()

    if len(loan_1000_chunk) > 0:
        loan_1000_parts.append(
            loan_1000_chunk.drop(columns=["_income_numeric", "_loan_numeric"])
        )

    # -----------------------------------------------------
    # C. Top 100 applicant income rows
    # -----------------------------------------------------
    candidate_top_income = pd.concat(
        [top_income_df, chunk],
        ignore_index=True
    )

    candidate_top_income["_income_numeric"] = safe_to_numeric(
        candidate_top_income[income_col]
    )

    top_income_df = (
        candidate_top_income
        .dropna(subset=["_income_numeric"])
        .nlargest(TOP_N, "_income_numeric")
        .reset_index(drop=True)
    )

    # -----------------------------------------------------
    # D. Top 100 loan amount rows
    # -----------------------------------------------------
    candidate_top_loan = pd.concat(
        [top_loan_df, chunk],
        ignore_index=True
    )

    candidate_top_loan["_loan_numeric"] = safe_to_numeric(
        candidate_top_loan[loan_col]
    )

    top_loan_df = (
        candidate_top_loan
        .dropna(subset=["_loan_numeric"])
        .nlargest(TOP_N, "_loan_numeric")
        .reset_index(drop=True)
    )

    print(
        f"Processed chunk {chunk_id} | "
        f"Total rows processed: {total_rows_processed:,} | "
        f"Income=$1,000 rows found so far: {sum(len(x) for x in income_1000_parts):,} | "
        f"Loan=$1,000 rows found so far: {sum(len(x) for x in loan_1000_parts):,}"
    )


# =========================================================
# 4. Build overall distribution table
# =========================================================

overall_distribution = (
    pd.Series(overall_target_counts)
    .sort_index()
    .reset_index()
)

overall_distribution.columns = [class_target_col, "count"]
overall_distribution["percent"] = (
    overall_distribution["count"] / overall_distribution["count"].sum() * 100
).round(2)

# Make sure both classes exist in the table
overall_distribution = (
    pd.DataFrame({class_target_col: ["0", "1"]})
    .merge(overall_distribution, on=class_target_col, how="left")
    .fillna(0)
)

print("\nOverall loan_approved distribution in full dataset:")
display(overall_distribution)


# =========================================================
# 5. Combine final groups
# =========================================================

if income_1000_parts:
    income_1000_df = pd.concat(income_1000_parts, ignore_index=True)
else:
    income_1000_df = pd.DataFrame(columns=[income_col, loan_col, class_target_col])

if loan_1000_parts:
    loan_1000_df = pd.concat(loan_1000_parts, ignore_index=True)
else:
    loan_1000_df = pd.DataFrame(columns=[income_col, loan_col, class_target_col])

helper_cols = ["_income_numeric", "_loan_numeric"]

top_income_df_clean = top_income_df.drop(
    columns=[col for col in helper_cols if col in top_income_df.columns],
    errors="ignore"
)

top_loan_df_clean = top_loan_df.drop(
    columns=[col for col in helper_cols if col in top_loan_df.columns],
    errors="ignore"
)


# =========================================================
# 6. Final reports
# =========================================================

print("\nFinal analysis done.")
print("Total rows processed:", f"{total_rows_processed:,}")

make_full_report(
    "Group 1: Rows where applicant_income_000s = 1, meaning applicant income is $1,000",
    income_1000_df,
    overall_distribution
)

make_full_report(
    "Group 2: Rows where loan_amount_000s = 1, meaning loan amount is $1,000",
    loan_1000_df,
    overall_distribution
)

make_full_report(
    "Group 3: Top 100 rows with the highest applicant_income_000s",
    top_income_df_clean,
    overall_distribution
)

make_full_report(
    "Group 4: Top 100 rows with the highest loan_amount_000s",
    top_loan_df_clean,
    overall_distribution
)

## Step 11: Explore Features and Approval Rate

This step takes a random sample of 20,000 rows and studies how each feature relates to loan approval. Categorical features are grouped by their most common values, and numeric features are grouped into bins. The output shows approval and rejection rates for each group and highlights groups that are far from the overall approval rate.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# =========================================================
# 0. Settings
# =========================================================

DATA_PATH = Path("hmda_2017_classification_ready.csv")

CHUNK_SIZE = 200_000
SAMPLE_SIZE = 20_000
RANDOM_STATE = 42

TARGET_COL = "loan_approved"

TOP_N_CATEGORIES = 15
NUMERIC_BINS = 10

# Continuous / semi-continuous numeric columns
continuous_numeric_cols = [
    "loan_amount_000s",
    "applicant_income_000s",
    "population",
    "minority_population",
    "hud_median_family_income",
    "tract_to_msamd_income",
    "number_of_owner_occupied_units",
    "number_of_1_to_4_family_units"
]

# Columns that should not be analyzed as features
columns_to_skip = [
    TARGET_COL,
    "action_taken",
    "action_taken_name"
]

# =========================================================
# 1. Feature explanations
# =========================================================

feature_explanations = {
    "as_of_year": "Report year. Usually constant in this dataset.",
    "respondent_id": "Financial institution / lender identifier.",
    "agency_name": "Name of the supervising agency.",
    "agency_abbr": "Abbreviation of the supervising agency.",
    "agency_code": "Code of the supervising agency.",
    "loan_type_name": "Loan type label, such as Conventional, FHA, VA, or FSA/RHS.",
    "loan_type": "Loan type code.",
    "property_type_name": "Property type label.",
    "property_type": "Property type code.",
    "loan_purpose_name": "Loan purpose label: home purchase, refinancing, or home improvement.",
    "loan_purpose": "Loan purpose code.",
    "owner_occupancy_name": "Whether the property is owner-occupied.",
    "owner_occupancy": "Owner occupancy code.",
    "loan_amount_000s": "Loan amount in thousands of dollars.",
    "preapproval_name": "Preapproval status label.",
    "preapproval": "Preapproval status code.",
    "msamd_name": "Metropolitan statistical area / division name.",
    "msamd": "Metropolitan statistical area / division code.",
    "state_name": "State name.",
    "state_abbr": "State abbreviation.",
    "state_code": "State code.",
    "county_name": "County name.",
    "county_code": "County code.",
    "census_tract_number": "Census tract identifier.",
    "applicant_ethnicity_name": "Applicant ethnicity label.",
    "applicant_ethnicity": "Applicant ethnicity code.",
    "co_applicant_ethnicity_name": "Co-applicant ethnicity label.",
    "co_applicant_ethnicity": "Co-applicant ethnicity code.",
    "applicant_race_name_1": "Main applicant race label.",
    "applicant_race_1": "Main applicant race code.",
    "co_applicant_race_name_1": "Main co-applicant race label.",
    "co_applicant_race_1": "Main co-applicant race code.",
    "applicant_sex_name": "Applicant sex label.",
    "applicant_sex": "Applicant sex code.",
    "co_applicant_sex_name": "Co-applicant sex label.",
    "co_applicant_sex": "Co-applicant sex code.",
    "applicant_income_000s": "Applicant annual income in thousands of dollars.",
    "hoepa_status_name": "Whether the loan is a HOEPA high-cost loan.",
    "hoepa_status": "HOEPA status code.",
    "lien_status_name": "Lien status label.",
    "lien_status": "Lien status code.",
    "population": "Population of the census tract.",
    "minority_population": "Minority population percentage in the census tract.",
    "hud_median_family_income": "Median family income in the area.",
    "tract_to_msamd_income": "Tract income as a percentage of MSA/MD income.",
    "number_of_owner_occupied_units": "Number of owner-occupied housing units in the census tract.",
    "number_of_1_to_4_family_units": "Number of 1-to-4 family housing units in the census tract.",
    "loan_approved": "Classification target: 1 means approved/accepted, 0 means rejected."
}

# =========================================================
# 2. Take an in-memory random sample from the full dataset
# =========================================================

sample_df = None
total_rows_seen = 0

for chunk_id, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    total_rows_seen += len(chunk)

    # Strip whitespace from all values
    for col in chunk.columns:
        chunk[col] = chunk[col].astype(str).str.strip()

    # Keep a running in-memory random sample
    if sample_df is None:
        sample_df = chunk.copy()
    else:
        sample_df = pd.concat([sample_df, chunk], ignore_index=True)

    # Downsample if the temporary sample becomes larger than needed
    if len(sample_df) > SAMPLE_SIZE:
        sample_df = sample_df.sample(
            n=SAMPLE_SIZE,
            random_state=RANDOM_STATE + chunk_id
        ).reset_index(drop=True)

    print(
        f"Processed chunk {chunk_id} | "
        f"Rows seen: {total_rows_seen:,} | "
        f"Current sample size: {len(sample_df):,}"
    )

# Final shuffle
df_sample = sample_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

print("\nFinal in-memory sample shape:", df_sample.shape)
display(df_sample.head())

# =========================================================
# 3. Prepare target distribution
# =========================================================

if TARGET_COL not in df_sample.columns:
    raise ValueError(f"Target column '{TARGET_COL}' was not found.")

df_sample[TARGET_COL] = df_sample[TARGET_COL].astype(str).str.strip()

overall_counts = df_sample[TARGET_COL].value_counts(dropna=False).sort_index()
overall_percent = (overall_counts / overall_counts.sum() * 100).round(2)

overall_target_distribution = pd.DataFrame({
    TARGET_COL: overall_counts.index,
    "overall_count": overall_counts.values,
    "overall_percent": overall_percent.values
})

print("\nOverall target distribution in the 20,000-row sample:")
display(overall_target_distribution)

overall_approval_rate = pd.to_numeric(
    df_sample[TARGET_COL],
    errors="coerce"
).mean()

overall_rejection_rate = 1 - overall_approval_rate

print("Overall approval rate:", round(overall_approval_rate * 100, 2), "%")
print("Overall rejection rate:", round(overall_rejection_rate * 100, 2), "%")

# =========================================================
# 4. Helper functions
# =========================================================

def is_numeric_feature(df, col, numeric_cols):
    """
    Decide whether a feature should be treated as numeric.
    Some code columns look numeric but are actually categorical.
    """
    if col in numeric_cols:
        return True

    numeric_s = pd.to_numeric(df[col], errors="coerce")
    numeric_ratio = numeric_s.notna().mean()

    return numeric_ratio > 0.95 and df[col].nunique(dropna=False) > 20


def make_binned_numeric_feature(df, col, n_bins=10):
    """
    Convert a numeric column into quantile bins.
    This makes target distribution easier to compare.
    """
    numeric_s = pd.to_numeric(df[col], errors="coerce")

    if numeric_s.nunique(dropna=True) < 3:
        return numeric_s.astype(str)

    try:
        binned = pd.qcut(
            numeric_s,
            q=n_bins,
            duplicates="drop"
        )
        return binned.astype(str)
    except Exception:
        return numeric_s.astype(str)


def make_categorical_feature_limited(df, col, top_n=15):
    """
    Keep only top N categories.
    Less frequent categories are grouped as Other.
    """
    s = df[col].astype(str).str.strip()

    top_categories = s.value_counts(dropna=False).head(top_n).index

    limited = s.where(s.isin(top_categories), other="Other")

    return limited


def target_distribution_by_feature(
    df,
    feature_col,
    target_col=TARGET_COL,
    numeric_cols=continuous_numeric_cols,
    top_n_categories=15,
    numeric_bins=10
):
    """
    Create one target distribution table for one feature.
    Numeric features are binned.
    Categorical features are limited to top categories.
    """
    temp = df[[feature_col, target_col]].copy()

    temp[target_col] = temp[target_col].astype(str).str.strip()

    if is_numeric_feature(temp, feature_col, numeric_cols):
        temp["feature_group"] = make_binned_numeric_feature(
            temp,
            feature_col,
            n_bins=numeric_bins
        )
        feature_type = "numeric_binned"
    else:
        temp["feature_group"] = make_categorical_feature_limited(
            temp,
            feature_col,
            top_n=top_n_categories
        )
        feature_type = "categorical_limited"

    grouped = (
        temp
        .groupby(["feature_group", target_col])
        .size()
        .reset_index(name="count")
    )

    pivot = grouped.pivot_table(
        index="feature_group",
        columns=target_col,
        values="count",
        fill_value=0
    ).reset_index()

    if "0" not in pivot.columns:
        pivot["0"] = 0

    if "1" not in pivot.columns:
        pivot["1"] = 0

    pivot = pivot.rename(columns={
        "0": "rejected_count",
        "1": "approved_count"
    })

    pivot["total_count"] = pivot["rejected_count"] + pivot["approved_count"]

    pivot["approved_percent"] = (
        pivot["approved_count"] / pivot["total_count"] * 100
    ).round(2)

    pivot["rejected_percent"] = (
        pivot["rejected_count"] / pivot["total_count"] * 100
    ).round(2)

    pivot["overall_approved_percent"] = round(overall_approval_rate * 100, 2)
    pivot["overall_rejected_percent"] = round(overall_rejection_rate * 100, 2)

    pivot["approved_percent_minus_overall"] = (
        pivot["approved_percent"] - pivot["overall_approved_percent"]
    ).round(2)

    pivot["feature"] = feature_col
    pivot["feature_type"] = feature_type
    pivot["feature_explanation"] = feature_explanations.get(
        feature_col,
        "No manual explanation provided for this feature."
    )

    pivot = pivot[
        [
            "feature",
            "feature_explanation",
            "feature_type",
            "feature_group",
            "total_count",
            "approved_count",
            "rejected_count",
            "approved_percent",
            "rejected_percent",
            "overall_approved_percent",
            "overall_rejected_percent",
            "approved_percent_minus_overall"
        ]
    ]

    pivot = pivot.sort_values(
        "total_count",
        ascending=False
    ).reset_index(drop=True)

    return pivot


# =========================================================
# 5. Show target distribution for each feature
# =========================================================

feature_cols = [
    col for col in df_sample.columns
    if col not in columns_to_skip
]

all_feature_tables = {}

for col in feature_cols:
    print("=" * 120)
    print(f"Feature: {col}")
    print("Explanation:", feature_explanations.get(
        col,
        "No manual explanation provided for this feature."
    ))

    feature_table = target_distribution_by_feature(
        df_sample,
        feature_col=col,
        target_col=TARGET_COL,
        numeric_cols=continuous_numeric_cols,
        top_n_categories=TOP_N_CATEGORIES,
        numeric_bins=NUMERIC_BINS
    )

    all_feature_tables[col] = feature_table

    display(feature_table)

# =========================================================
# 6. Optional: one combined dataframe inside memory only
# =========================================================

all_feature_distributions_df = pd.concat(
    all_feature_tables.values(),
    ignore_index=True
)

print("=" * 120)
print("Combined in-memory table shape:", all_feature_distributions_df.shape)
display(all_feature_distributions_df.head(50))

# =========================================================
# 7. Optional: show strongest differences from overall approval rate
# =========================================================

important_groups = all_feature_distributions_df[
    all_feature_distributions_df["total_count"] >= 50
].copy()

important_groups["abs_difference"] = (
    important_groups["approved_percent_minus_overall"]
    .abs()
)

important_groups = important_groups.sort_values(
    "abs_difference",
    ascending=False
).reset_index(drop=True)

print("=" * 120)
print("Feature groups with strongest difference from overall approval rate:")
display(important_groups.head(50))

## Step 12: Create the Final Modeling Dataset

This step removes columns that are redundant, coded duplicates, post-decision fields, or not needed for modeling. It keeps clear label columns, useful numeric columns, location information, and the final target column. The output file is `hmda_2017_ready.csv`, with 29 columns and 8,064,813 rows.

In [ ]:
import pandas as pd
from pathlib import Path

# =========================================================
# 0. File paths and settings
# =========================================================

DATA_PATH = Path("hmda_2017_classification_ready_cleaned_invalid_removed.csv")
OUTPUT_PATH = Path("hmda_2017_ready.csv")

CHUNK_SIZE = 200_000

# =========================================================
# 1. Columns to drop
# =========================================================

columns_to_drop = [
    "agency_abbr",
    "agency_code",
    "loan_type",
    "property_type",
    "loan_purpose",
    "owner_occupancy",
    "preapproval",
    "msamd",
    "state_abbr",
    "applicant_ethnicity",
    "co_applicant_ethnicity",
    "applicant_race_1",
    "co_applicant_race_1",
    "applicant_sex",
    "co_applicant_sex",
    "purchaser_type_name",
    "purchaser_type",
    "hoepa_status_name",
    "hoepa_status",
    "lien_status",
    "action_taken",
    "action_taken_name",
    "as_of_year"
]

# =========================================================
# 2. Check which columns exist
# =========================================================

header_df = pd.read_csv(
    DATA_PATH,
    nrows=0,
    dtype=str,
    keep_default_na=False,
    na_filter=False
)

existing_columns_to_drop = [
    col for col in columns_to_drop
    if col in header_df.columns
]

missing_columns_to_drop = [
    col for col in columns_to_drop
    if col not in header_df.columns
]

print("Original column count:", len(header_df.columns))
print("Columns found and will be dropped:", len(existing_columns_to_drop))
print("Expected final column count:", len(header_df.columns) - len(existing_columns_to_drop))

print("\nColumns that will be dropped:")
for col in existing_columns_to_drop:
    print("-", col)

if missing_columns_to_drop:
    print("\nColumns not found in dataset:")
    for col in missing_columns_to_drop:
        print("-", col)

# =========================================================
# 3. Process file chunk by chunk
# =========================================================

first_chunk = True
total_rows_processed = 0

for chunk_id, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    rows_before = len(chunk)
    total_rows_processed += rows_before

    # Drop only columns that exist in this chunk
    drop_cols_in_chunk = [
        col for col in existing_columns_to_drop
        if col in chunk.columns
    ]

    chunk = chunk.drop(columns=drop_cols_in_chunk)

    # Write chunk to output file
    chunk.to_csv(
        OUTPUT_PATH,
        index=False,
        mode="w" if first_chunk else "a",
        header=first_chunk
    )

    first_chunk = False

    print(
        f"Chunk {chunk_id} | "
        f"Rows processed: {total_rows_processed:,} | "
        f"Current column count: {chunk.shape[1]}"
    )

print("\nDone.")
print("Output saved to:", OUTPUT_PATH.resolve())
print("Total rows processed:", f"{total_rows_processed:,}")

# =========================================================
# 4. Final column count check
# =========================================================

output_header_df = pd.read_csv(
    OUTPUT_PATH,
    nrows=0,
    dtype=str,
    keep_default_na=False,
    na_filter=False
)

print("\nFinal check:")
print("Output column count:", len(output_header_df.columns))
print("Output columns:")
print(output_header_df.columns.tolist())

## Step 13: Check Numeric Columns in the Final Dataset

This step creates the final numeric summary for `hmda_2017_ready.csv`. It checks the same main numeric columns after final column selection. This is a final quality check before sampling and model building.

In [ ]:
import pandas as pd
from pathlib import Path

# Input file
DATA_PATH = Path("hmda_2017_ready.csv")

CHUNK_SIZE = 200_000

# Continuous / semi-continuous numeric columns
continuous_numeric_cols = [
    "loan_amount_000s",
    "applicant_income_000s",
    "population",
    "minority_population",
    "hud_median_family_income",
    "tract_to_msamd_income",
    "number_of_owner_occupied_units",
    "number_of_1_to_4_family_units"
]

# --------------------------------------------------
# 1. Check which columns exist in the file
# --------------------------------------------------
header_df = pd.read_csv(
    DATA_PATH,
    nrows=0,
    dtype=str,
    keep_default_na=False,
    na_filter=False
)

existing_numeric_cols = [
    col for col in continuous_numeric_cols
    if col in header_df.columns
]

missing_numeric_cols = [
    col for col in continuous_numeric_cols
    if col not in header_df.columns
]

print("Existing numeric columns:")
print(existing_numeric_cols)

if missing_numeric_cols:
    print("\nColumns not found:")
    print(missing_numeric_cols)

# --------------------------------------------------
# 2. Read the file chunk by chunk
#    Store only numeric columns, then make one final summary
# --------------------------------------------------
numeric_chunks = []

total_rows = 0

for chunk_id, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    total_rows += len(chunk)

    # Convert selected columns to numeric
    numeric_chunk = chunk[existing_numeric_cols].apply(
        pd.to_numeric,
        errors="coerce"
    )

    numeric_chunks.append(numeric_chunk)

    print(f"Processed chunk {chunk_id} | Total rows processed: {total_rows:,}")

# --------------------------------------------------
# 3. Combine numeric chunks and create one final summary
# --------------------------------------------------
df_numeric_all = pd.concat(numeric_chunks, ignore_index=True)

numeric_summary = df_numeric_all.describe(
    percentiles=[0.25, 0.5, 0.75]
).T

numeric_summary = numeric_summary[
    ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
].rename(columns={
    "count": "non_missing_count",
    "mean": "mean",
    "std": "std",
    "min": "min",
    "25%": "q1_25%",
    "50%": "median_50%",
    "75%": "q3_75%",
    "max": "max"
})

# Add missing counts
numeric_summary["missing_count"] = df_numeric_all.isna().sum()
numeric_summary["missing_percent"] = (
    df_numeric_all.isna().mean() * 100
).round(2)

# Reorder columns
numeric_summary = numeric_summary[
    [
        "non_missing_count",
        "missing_count",
        "missing_percent",
        "min",
        "q1_25%",
        "median_50%",
        "mean",
        "q3_75%",
        "max",
        "std"
    ]
]

# Optional: round values for readability
numeric_summary = numeric_summary.round(3)

print("\nTotal rows processed:", f"{total_rows:,}")
display(numeric_summary)

## Step 14: Create Final Samples for Modeling

This step creates two 500,000-row datasets. The classification sample keeps the same approval and rejection ratio as the full dataset, so it is stratified by `loan_approved`. The regression sample keeps only approved loans, because the regression target is loan amount for approved applications. Both samples are saved as CSV files for the next modeling stage.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# 0. File paths and settings
# =========================================================

# Change this path if your final cleaned dataset has another name
DATA_PATH = Path("hmda_2017_ready.csv")

CLASSIFICATION_OUTPUT_PATH = Path("hmda_classification_stratified_500k.csv")
REGRESSION_OUTPUT_PATH = Path("hmda_regression_approved_500k.csv")

CHUNK_SIZE = 200_000
RANDOM_STATE = 42

CLASSIFICATION_SAMPLE_SIZE = 500_000
REGRESSION_SAMPLE_SIZE = 500_000

TARGET_COL = "loan_approved"

# =========================================================
# 1. First pass: count target distribution in full dataset
# =========================================================

target_counts = {}
total_rows = 0

for chunk_id, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    if TARGET_COL not in chunk.columns:
        raise ValueError(f"Target column '{TARGET_COL}' not found in dataset.")

    chunk[TARGET_COL] = chunk[TARGET_COL].astype(str).str.strip()

    counts = chunk[TARGET_COL].value_counts(dropna=False)

    for key, value in counts.items():
        target_counts[key] = target_counts.get(key, 0) + int(value)

    total_rows += len(chunk)

    print(f"Counting pass | Chunk {chunk_id} | Rows counted: {total_rows:,}")

target_counts = pd.Series(target_counts).sort_index()

print("\nFull dataset target distribution:")
display(target_counts)

target_percent = (target_counts / target_counts.sum() * 100).round(2)

print("\nFull dataset target distribution percent:")
display(target_percent)

# =========================================================
# 2. Compute sample size per class for stratified classification sample
# =========================================================

if target_counts.sum() < CLASSIFICATION_SAMPLE_SIZE:
    raise ValueError("Classification sample size is larger than total dataset size.")

target_sample_sizes = (
    target_counts / target_counts.sum() * CLASSIFICATION_SAMPLE_SIZE
).round().astype(int)

# Fix possible rounding difference
rounding_diff = CLASSIFICATION_SAMPLE_SIZE - target_sample_sizes.sum()

if rounding_diff != 0:
    largest_class = target_counts.idxmax()
    target_sample_sizes.loc[largest_class] += rounding_diff

print("\nTarget sample sizes per class:")
display(target_sample_sizes)

if "1" not in target_counts.index:
    raise ValueError("No approved rows found: loan_approved = 1 does not exist.")

if target_counts.loc["1"] < REGRESSION_SAMPLE_SIZE:
    raise ValueError(
        f"Not enough approved rows. Available approved rows: {target_counts.loc['1']:,}"
    )

# =========================================================
# 3. Second pass: create stratified classification sample
# =========================================================

classification_samples = {
    class_label: pd.DataFrame()
    for class_label in target_sample_sizes.index
}

for chunk_id, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    chunk[TARGET_COL] = chunk[TARGET_COL].astype(str).str.strip()

    for class_label, desired_n in target_sample_sizes.items():
        class_chunk = chunk[chunk[TARGET_COL] == class_label].copy()

        if len(class_chunk) == 0:
            continue

        classification_samples[class_label] = pd.concat(
            [classification_samples[class_label], class_chunk],
            ignore_index=True
        )

        if len(classification_samples[class_label]) > desired_n:
            classification_samples[class_label] = (
                classification_samples[class_label]
                .sample(
                    n=desired_n,
                    random_state=RANDOM_STATE + chunk_id + int(class_label)
                )
                .reset_index(drop=True)
            )

    print(
        f"Classification sampling pass | Chunk {chunk_id} | "
        + " | ".join([
            f"class {label}: {len(df):,}/{target_sample_sizes[label]:,}"
            for label, df in classification_samples.items()
        ])
    )

df_classification_sample = pd.concat(
    classification_samples.values(),
    ignore_index=True
)

df_classification_sample = df_classification_sample.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

df_classification_sample.to_csv(
    CLASSIFICATION_OUTPUT_PATH,
    index=False
)

print("\nClassification sample saved.")
print("Path:", CLASSIFICATION_OUTPUT_PATH.resolve())
print("Shape:", df_classification_sample.shape)

print("\nClassification sample target distribution:")
display(df_classification_sample[TARGET_COL].value_counts(dropna=False).sort_index())

print("\nClassification sample target distribution percent:")
display(
    (df_classification_sample[TARGET_COL].value_counts(normalize=True).sort_index() * 100)
    .round(2)
)

# =========================================================
# 4. Third pass: create approved-only regression sample
# =========================================================

regression_sample = pd.DataFrame()
approved_rows_seen = 0

for chunk_id, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    chunk[TARGET_COL] = chunk[TARGET_COL].astype(str).str.strip()

    approved_chunk = chunk[chunk[TARGET_COL] == "1"].copy()

    if len(approved_chunk) == 0:
        continue

    approved_rows_seen += len(approved_chunk)

    regression_sample = pd.concat(
        [regression_sample, approved_chunk],
        ignore_index=True
    )

    if len(regression_sample) > REGRESSION_SAMPLE_SIZE:
        regression_sample = (
            regression_sample
            .sample(
                n=REGRESSION_SAMPLE_SIZE,
                random_state=RANDOM_STATE + chunk_id
            )
            .reset_index(drop=True)
        )

    print(
        f"Regression sampling pass | Chunk {chunk_id} | "
        f"Approved rows seen: {approved_rows_seen:,} | "
        f"Current regression sample size: {len(regression_sample):,}"
    )

regression_sample = regression_sample.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

regression_sample.to_csv(
    REGRESSION_OUTPUT_PATH,
    index=False
)

print("\nRegression approved-only sample saved.")
print("Path:", REGRESSION_OUTPUT_PATH.resolve())
print("Shape:", regression_sample.shape)

print("\nRegression sample loan_approved distribution:")
display(regression_sample[TARGET_COL].value_counts(dropna=False).sort_index())

# =========================================================
# 5. Final checks
# =========================================================

df_cls_check = pd.read_csv(
    CLASSIFICATION_OUTPUT_PATH,
    dtype=str,
    keep_default_na=False,
    na_filter=False
)

df_reg_check = pd.read_csv(
    REGRESSION_OUTPUT_PATH,
    dtype=str,
    keep_default_na=False,
    na_filter=False
)

print("\nFinal check - Classification sample shape:")
print(df_cls_check.shape)

print("\nFinal check - Classification target distribution:")
display(df_cls_check[TARGET_COL].value_counts(dropna=False).sort_index())

print("\nFinal check - Classification target distribution percent:")
display(
    (df_cls_check[TARGET_COL].value_counts(normalize=True).sort_index() * 100)
    .round(2)
)

print("\nFinal check - Regression sample shape:")
print(df_reg_check.shape)

print("\nFinal check - Regression target distribution:")
display(df_reg_check[TARGET_COL].value_counts(dropna=False).sort_index())